# N-Gram Language Models
In this exercise, we will use n-gram language models to predict the probability of text, and generate it.

In [1]:
import nltk
from nltk.corpus import gutenberg

First, we load Jane Austen's *Emma* from NLTK's gutenberg corpus that we also used in a previous exercise. Tokenize and lowercase this text such that we have a list of words.

In [2]:
raw_text = gutenberg.raw('austen-emma.txt')
words = [w.lower() for w in nltk.word_tokenize(raw_text)]
print(len(words))

191855


Write an n-gram language model class that takes the word list and a parameter `n` as inputs, where `n` is a positive integer larger than 1 that determines the `n` of the n-gram LM. The LM should build a dictionary of n-gram counts from the word list.

In [3]:
from collections import defaultdict

class NGramLanguageModel:
    
    def __init__(self, words, n=2):
        assert n > 1, "n needs to be a positive integer > 1"
        assert n <= len(words), "n can't be larger than the number of words"
        self.n = n
        self.counts = defaultdict(int)
        for i in range(len(words) - self.n + 1):
            ngram = tuple(words[i:i+self.n])
            self.counts[ngram] += 1
            self.counts[ngram[:-1]] += 1
        self.counts[ngram[1:]] += 1  # last n-gram in the input

Now we "train" the n-gram LM by building the n-gram counts of the Emma novel. Use a low `n` (i.e. 2 or 3).

In [4]:
ngrams = NGramLanguageModel(words, n=3)

Let's add a method `log_probability` to the n-gram LM class that computes the probability of an input string. Since multiplying many probabilities (<= 1) results in very small numbers that can underflow, we sum the log probabilities instead. For simplicity, ignore unknown n-grams (we would use smoothing or backoff) and start with `p(w_n | w_1, ..., w_{n-1})`.

In [5]:
import math

def log_probability(self, input_string):
    log_prob = []
    words = [w.lower() for w in nltk.word_tokenize(input_string)]
    for i in range(len(words) - self.n + 1):
        ngram = tuple(words[i:i+self.n])
        if ngram in self.counts:
            prob = self.counts[ngram] / self.counts[ngram[:-1]]
            log_prob.append(math.log(prob))
    return sum(log_prob)

NGramLanguageModel.log_probability = log_probability

In [6]:
print(ngrams.log_probability('There is a house in New Orleans.'))
print(ngrams.log_probability('There is a house in New Orleans. '
                             'And then there are also many other houses in many other cities.'))

-2.0402208285265546
-8.864594498569641


Shorter texts will have higher log probability than longer texts, so we need to normalize it by the number of words in the input string.

In [7]:
import numpy as np

def log_probability(self, input_string):
    log_prob = []
    words = [w.lower() for w in nltk.word_tokenize(input_string)]
    for i in range(len(words) - self.n + 1):
        ngram = tuple(words[i:i+self.n])
        if ngram in self.counts:
            prob = self.counts[ngram] / self.counts[ngram[:-1]]
            log_prob.append(math.log(prob))
    return np.mean(log_prob)

NGramLanguageModel.log_probability = log_probability

Lets predict the probabilities of two novels under our trained model: Jane Austen's *Sense and Sensibility* (`austen-sense.txt`) and Shakespeare's *Hamlet* (`shakespeare-hamlet.txt`).
- What do you expect will happen?
- What do you observe?

In [8]:
austen_sense = gutenberg.raw('austen-sense.txt')
shakespeare_hamlet = gutenberg.raw('shakespeare-hamlet.txt')
print(ngrams.log_probability('There is a house in New Orleans.'))
print(ngrams.log_probability(raw_text))
print(ngrams.log_probability(austen_sense))
print(ngrams.log_probability(shakespeare_hamlet))

-2.0402208285265546
-1.749210563827336
-2.405643228181783
-2.789276286803534


How many n-grams are known in each input?

In [9]:
def known_ngrams(self, input_string):
    known = 0
    words = [w.lower() for w in nltk.word_tokenize(input_string)]
    for i in range(len(words) - self.n + 1):
        ngram = tuple(words[i:i+self.n])
        if ngram in self.counts:
            known += 1
    return known, len(words) - self.n + 1

NGramLanguageModel.known_ngrams = known_ngrams
known, total = ngrams.known_ngrams(raw_text)
print(f'Known n-grams in Emma: {known/total:.2%} ({known}/{total})')
known, total = ngrams.known_ngrams(austen_sense)
print(f'Known n-grams in Sense and Sensibility: {known/total:.2%} ({known}/{total})')
known, total = ngrams.known_ngrams(shakespeare_hamlet)
print(f'Known n-grams in Hamlet: {known/total:.2%} ({known}/{total})')

Known n-grams in Emma: 100.00% (191853/191853)
Known n-grams in Sense and Sensibility: 29.56% (41806/141438)
Known n-grams in Hamlet: 8.82% (3211/36409)


Let's add a method `generate` that takes the start of a sentence ("prompt") and a number of words to generate, then continues our prompt.

In [10]:
def generate(self, prompt, num_words=10):
    words = [w.lower() for w in nltk.word_tokenize(prompt)]
    for i in range(num_words):
        prefix = tuple(words[-(self.n - 1):])
        if prefix not in self.counts:
            words.append('[END]')
            break
        next_word_dist = {}
        for ngram in self.counts:
            if len(ngram) == self.n and ngram[:-1] == prefix:
                next_word_dist[ngram] = self.counts[ngram] / self.counts[prefix]
        # print(prefix, next_word_dist)
        best_ngram, prob = max(next_word_dist.items(), key=lambda x: x[1])
        print(best_ngram, prob)
        words.append(best_ngram[-1])
    return ' '.join(words)

NGramLanguageModel.generate = generate

Play around with a few different prompts.

In [11]:
ngrams.generate("I went for a walk", num_words=20)

('a', 'walk', ',') 0.25
('walk', ',', 'or') 0.1111111111111111
(',', 'or', 'any') 0.07103825136612021
('or', 'any', 'thing') 0.35294117647058826
('any', 'thing', 'to') 0.08536585365853659
('thing', 'to', 'be') 0.25925925925925924
('to', 'be', 'sure') 0.04132231404958678
('be', 'sure', ',') 0.23076923076923078
('sure', ',', "''") 0.18181818181818182
(',', "''", 'said') 0.3852739726027397
("''", 'said', 'he') 0.2188679245283019
('said', 'he', ',') 0.5797101449275363
('he', ',', '``') 0.35714285714285715
(',', '``', 'i') 0.152
('``', 'i', 'am') 0.16091954022988506
('i', 'am', 'sure') 0.2715736040609137
('am', 'sure', 'i') 0.16822429906542055
('sure', 'i', 'should') 0.19047619047619047
('i', 'should', 'not') 0.21100917431192662
('should', 'not', 'have') 0.35


"i went for a walk , or any thing to be sure , '' said he , `` i am sure i should not have"